# 08 — Planar array analysis

Apply a joint three-channel plane-wave coherence search to the fixed
baseline-corrected event Streams produced by Notebook 07. A permissive analysis
tier retains weak candidates for testing, while a stricter core tier is
recorded descriptively. Continuous coherence, correlation, surface-width, and
window-stability metrics are retained for every analyzed event.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from modules import project_config as config

OUTPUT_ROOT = config.OUTPUT_DIR / "array_analysis_planar_refined"
LAST_OUTPUT_ROOT = config.OUTPUT_DIR / "candidate_airwave_preparation"
DERIVED_DIR = LAST_OUTPUT_ROOT / "derived"
EVENT_STREAMS_DIR = LAST_OUTPUT_ROOT / "event_streams"

ANALYSIS_CONFIG_FILE = config.OUTPUT_DIR / "02_analysis_configuration.json"
GEOMETRY_FILE = config.OUTPUT_DIR / "02_bchh_geometry.csv"
WEATHER_SUMMARY_FILE = config.OUTPUT_DIR / "03_weather_acoustic_summary.csv"

for required_file in (
    ANALYSIS_CONFIG_FILE,
    GEOMETRY_FILE,
    WEATHER_SUMMARY_FILE,
):
    if not required_file.exists():
        raise FileNotFoundError(required_file)

analysis_config = json.loads(ANALYSIS_CONFIG_FILE.read_text())
weather_summary = pd.read_csv(WEATHER_SUMMARY_FILE).iloc[0]

CANDIDATE_CATALOGUE_OVERRIDE: Path | None = None
EVENT_STREAM_DIR_OVERRIDE: Path | None = None
GEOMETRY_FILE_OVERRIDE: Path | None = None

# Permissive eligibility: analyze weak candidates rather than rejecting them before
# the array has a chance to test coherence.
ANALYSIS_MIN_STACK_PEAK_TO_PEAK_PA = 10.0
ANALYSIS_MIN_STACK_SNR = 1.5
ANALYSIS_MIN_CHANNEL_COUNT = 2
REQUIRE_VALID_MEASUREMENT_FOR_ANALYSIS = True

# Preserve the original thresholds as a descriptive core-catalogue label.
CORE_MIN_STACK_PEAK_TO_PEAK_PA = 25.0
CORE_MIN_STACK_SNR = 3.0
CORE_MIN_CHANNEL_COUNT = 2

# Named complex events bypass the short-pulse morphology requirement.
PRINCIPAL_EXPLOSION_EVENT_NUMBER = 13
FORCE_INCLUDE_EVENT_NUMBERS = (PRINCIPAL_EXPLOSION_EVENT_NUMBER,)
EVENT_LABELS = {PRINCIPAL_EXPLOSION_EVENT_NUMBER: "principal_first_stage_explosion"}

# Coarse plane-wave search: array -> source back azimuth, clockwise from north.
BACK_AZIMUTH_MIN_DEG = 180.0
BACK_AZIMUTH_MAX_DEG = 220.0
BACK_AZIMUTH_STEP_DEG = 0.5
APPARENT_SPEED_MIN_MPS = 280.0
APPARENT_SPEED_MAX_MPS = 440.0
APPARENT_SPEED_STEP_MPS = 2.0

# Event-specific refinement around the coarse maximum.
FINE_AZIMUTH_HALF_WIDTH_DEG = 2.0
FINE_AZIMUTH_STEP_DEG = 0.1
FINE_SPEED_HALF_WIDTH_MPS = 20.0
FINE_SPEED_STEP_MPS = 0.5

EXPECTED_SLC40_BACK_AZIMUTH_DEG = float(
    analysis_config["array_reference"]["back_azimuth_to_slc40_deg"]
)
PROVISIONAL_EFFECTIVE_SPEED_MPS = float(
    weather_summary["effective_sound_speed_mps"]
)
ARRAY_TO_SOURCE_REFERENCE_RANGE_M = float(
    analysis_config["array_reference"]["distance_m"]
)

# Event-specific score windows relative to the reduced/reference time (normally HD2).
DEFAULT_SCORE_WINDOW_RELATIVE_S = (-0.040, 0.180)
CUSTOM_SCORE_WINDOWS = {
    PRINCIPAL_EXPLOSION_EVENT_NUMBER: (-0.040, 0.350),
}
SCORE_WEIGHT_SEMBLANCE = 0.60
MAX_RESIDUAL_LAG_S = 0.040

# Descriptive near-maximum surface widths. These are not confidence intervals.
CONSTRAINT_SCORE_DROPS = (0.01, 0.02)

# Window-stability perturbations around each event's own scoring interval.
STABILITY_WINDOW_SHIFTS_S = (-0.008, 0.0, 0.008)
STABILITY_DURATION_FACTORS = (0.90, 1.0, 1.10)
STABILITY_LOCAL_AZIMUTH_HALF_WIDTH_DEG = 2.0
STABILITY_LOCAL_SPEED_HALF_WIDTH_MPS = 20.0
STABILITY_AZIMUTH_STEP_DEG = 0.1
STABILITY_SPEED_STEP_MPS = 0.5
RUN_WINDOW_STABILITY = True

# Provisional descriptive quality classes; retain continuous metrics in all exports.
QUALITY_A_MIN_SCORE = 0.90
QUALITY_A_MAX_AZIMUTH_STD_DEG = 0.5
QUALITY_A_MAX_SPEED_STD_MPS = 3.0
QUALITY_B_MIN_SCORE = 0.75
QUALITY_B_MAX_AZIMUTH_STD_DEG = 1.0
QUALITY_B_MAX_SPEED_STD_MPS = 8.0

# Figure display zoom. The broad search remains wider than this view.
DISPLAY_AZIMUTH_MIN_DEG = 195.0
DISPLAY_AZIMUTH_MAX_DEG = 207.0
DISPLAY_SPEED_MIN_MPS = 300.0
DISPLAY_SPEED_MAX_MPS = 400.0

MAX_EVENTS: int | None = None

FIGURE_DIR = OUTPUT_ROOT / "event_figures"
SURFACE_DIR = OUTPUT_ROOT / "surfaces"
COMPOSITE_DIR = OUTPUT_ROOT / "composites"
for directory in (OUTPUT_ROOT, FIGURE_DIR, SURFACE_DIR, COMPOSITE_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
from __future__ import annotations

import json
import math
import warnings
from pathlib import Path
from typing import Any, Iterable, Mapping

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize
from obspy import Stream, Trace, UTCDateTime, read
from scipy.signal import correlate, correlation_lags

CHANNELS = ("HD1", "HD2", "HD3")


def first_existing(paths: Iterable[Path], *, label: str) -> Path:
    checked = []
    for path in paths:
        path = Path(path).expanduser()
        checked.append(path)
        if path.exists():
            return path.resolve()
    lines = "\n".join(f"  {path}" for path in checked)
    raise FileNotFoundError(f"Could not locate {label}. Checked:\n{lines}")


def choose_column(frame: pd.DataFrame, candidates: Iterable[str], *, required=True):
    for name in candidates:
        if name in frame.columns:
            return name
    if required:
        raise KeyError(
            "None of the expected columns were found: "
            f"{list(candidates)}. Available columns: {frame.columns.tolist()}"
        )
    return None


def event_number_column(frame: pd.DataFrame) -> str:
    return choose_column(
        frame,
        ["event_number", "reconstructed_event_number", "candidate_event_number", "eventNumber"],
    )


def normalize_event_catalogue(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    source = event_number_column(out)
    if source != "event_number":
        out = out.rename(columns={source: "event_number"})
    out["event_number"] = pd.to_numeric(out["event_number"], errors="raise").astype(int)
    if out["event_number"].duplicated().any():
        duplicates = out.loc[out["event_number"].duplicated(), "event_number"].tolist()
        raise ValueError(f"Duplicate event numbers in candidate catalogue: {duplicates}")
    return out.sort_values("event_number").reset_index(drop=True)


def resolve_event_time_column(frame: pd.DataFrame) -> str:
    return choose_column(
        frame,
        [
            "reduced_event_epoch_s",
            "reduced_event_time_epoch_s",
            "event_time_epoch_s",
            "reference_time_epoch_s",
            "median_reduced_pick_epoch_s",
            "first_reduced_pick_epoch_s",
            "arrivalTimeEpochS",
        ],
    )


def locate_event_stream(event_row: pd.Series, event_stream_dir: Path) -> Path:
    for column in ("stream_file", "event_stream_file", "pickle_file", "stream_path"):
        if column in event_row.index and pd.notna(event_row[column]):
            candidate = Path(str(event_row[column])).expanduser()
            if not candidate.is_absolute():
                candidate = event_stream_dir / candidate
            if candidate.exists():
                return candidate.resolve()

    event_number = int(event_row["event_number"])
    patterns = [
        f"event_{event_number:03d}.pkl",
        f"event_{event_number:03d}_*.pkl",
        f"candidate_event_{event_number:03d}.pkl",
        f"*event*{event_number:03d}*.pkl",
    ]
    matches = []
    for pattern in patterns:
        matches.extend(event_stream_dir.glob(pattern))
    matches = sorted({path.resolve() for path in matches})
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one Pickle stream for event {event_number}; found {len(matches)}: {matches}"
        )
    return matches[0]


def load_event_stream(event_row: pd.Series, event_stream_dir: Path) -> tuple[Path, Stream]:
    path = locate_event_stream(event_row, event_stream_dir)
    stream = read(str(path), format="PICKLE")
    return path, stream


def unique_trace(stream: Stream, channel: str) -> Trace:
    matches = stream.select(channel=channel)
    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one {channel} trace; found {len(matches)}: {[tr.id for tr in matches]}"
        )
    return matches[0]


def robust_sigma(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    median = np.median(values)
    sigma = 1.4826 * np.median(np.abs(values - median))
    if sigma <= 0:
        sigma = float(np.std(values))
    return float(sigma)


def common_sampling_rate(stream: Stream, channels=CHANNELS) -> float:
    rates = [float(unique_trace(stream, channel).stats.sampling_rate) for channel in channels]
    if not np.allclose(rates, rates[0]):
        raise ValueError(f"Infrasound sampling rates differ: {rates}")
    return rates[0]


def interpolate_trace_epoch(trace: Trace, epoch_s: np.ndarray) -> np.ndarray:
    start = float(trace.stats.starttime.timestamp)
    sampling_rate = float(trace.stats.sampling_rate)
    source_epoch = start + np.arange(trace.stats.npts, dtype=float) / sampling_rate
    values = np.asarray(trace.data, dtype=float)
    return np.interp(epoch_s, source_epoch, values, left=np.nan, right=np.nan)


def normalized_rows(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    centered = values - np.nanmedian(values, axis=1, keepdims=True)
    rms = np.sqrt(np.nanmean(centered**2, axis=1, keepdims=True))
    rms[~np.isfinite(rms) | (rms <= 0)] = np.nan
    return centered / rms


def coherence_metrics(aligned: np.ndarray, *, score_weight_semblance: float) -> dict[str, float]:
    aligned = np.asarray(aligned, dtype=float)
    finite_columns = np.all(np.isfinite(aligned), axis=0)
    aligned = aligned[:, finite_columns]
    if aligned.shape[1] < 5:
        return {
            "semblance": np.nan,
            "mean_pairwise_correlation": np.nan,
            "median_pairwise_correlation": np.nan,
            "minimum_pairwise_correlation": np.nan,
            "coherence_score": np.nan,
            "n_samples": int(aligned.shape[1]),
        }

    normalized = normalized_rows(aligned)
    if not np.all(np.isfinite(normalized)):
        return {
            "semblance": np.nan,
            "mean_pairwise_correlation": np.nan,
            "median_pairwise_correlation": np.nan,
            "minimum_pairwise_correlation": np.nan,
            "coherence_score": np.nan,
            "n_samples": int(aligned.shape[1]),
        }

    n_channels = normalized.shape[0]
    numerator = np.sum(np.sum(normalized, axis=0) ** 2)
    denominator = n_channels * np.sum(normalized**2)
    semblance = float(numerator / denominator) if denominator > 0 else np.nan

    correlations = []
    for i in range(n_channels):
        for j in range(i + 1, n_channels):
            correlations.append(float(np.corrcoef(normalized[i], normalized[j])[0, 1]))
    correlations = np.asarray(correlations, dtype=float)
    mean_corr = float(np.nanmean(correlations))
    median_corr = float(np.nanmedian(correlations))
    minimum_corr = float(np.nanmin(correlations))
    correlation_01 = np.clip((mean_corr + 1.0) / 2.0, 0.0, 1.0)
    score = score_weight_semblance * semblance + (1.0 - score_weight_semblance) * correlation_01
    return {
        "semblance": semblance,
        "mean_pairwise_correlation": mean_corr,
        "median_pairwise_correlation": median_corr,
        "minimum_pairwise_correlation": minimum_corr,
        "coherence_score": float(score),
        "n_samples": int(aligned.shape[1]),
    }


def aligned_waveforms(
    stream: Stream,
    *,
    reference_epoch_s: float,
    relative_delays_s: Mapping[str, float],
    window_relative_s: tuple[float, float],
    channels=CHANNELS,
) -> tuple[np.ndarray, np.ndarray]:
    sampling_rate = common_sampling_rate(stream, channels)
    dt = 1.0 / sampling_rate
    relative_time = np.arange(window_relative_s[0], window_relative_s[1] + 0.5 * dt, dt)
    rows = []
    for channel in channels:
        trace = unique_trace(stream, channel)
        sample_epoch = reference_epoch_s + relative_time + float(relative_delays_s[channel])
        rows.append(interpolate_trace_epoch(trace, sample_epoch))
    values = np.vstack(rows)
    finite_columns = np.all(np.isfinite(values), axis=0)
    return relative_time[finite_columns], values[:, finite_columns]


def pairwise_diagnostics(aligned: np.ndarray, sampling_rate_hz: float, max_residual_lag_s: float):
    aligned = np.asarray(aligned, dtype=float)
    n = aligned.shape[0]
    corr_matrix = np.eye(n, dtype=float)
    lag_matrix = np.zeros((n, n), dtype=float)
    max_lag_samples = max(1, int(round(max_residual_lag_s * sampling_rate_hz)))

    for i in range(n):
        for j in range(i + 1, n):
            x = aligned[i] - np.nanmedian(aligned[i])
            y = aligned[j] - np.nanmedian(aligned[j])
            finite = np.isfinite(x) & np.isfinite(y)
            x = x[finite]
            y = y[finite]
            if len(x) < 5 or np.linalg.norm(x) == 0 or np.linalg.norm(y) == 0:
                corr = np.nan
                lag_s = np.nan
            else:
                cc = correlate(x, y, mode="full", method="auto")
                lags = correlation_lags(len(x), len(y), mode="full")
                keep = np.abs(lags) <= max_lag_samples
                cc = cc[keep] / (np.linalg.norm(x) * np.linalg.norm(y))
                lags = lags[keep]
                index = int(np.nanargmax(cc))
                corr = float(cc[index])
                lag_s = float(lags[index] / sampling_rate_hz)
            corr_matrix[i, j] = corr_matrix[j, i] = corr
            lag_matrix[i, j] = lag_s
            lag_matrix[j, i] = -lag_s

    closure_s = float(lag_matrix[0, 1] + lag_matrix[1, 2] + lag_matrix[2, 0])
    return corr_matrix, lag_matrix, closure_s


def polar_surface_plot(
    azimuth_deg: np.ndarray,
    speed_mps: np.ndarray,
    score_surface: np.ndarray,
    *,
    best_azimuth_deg: float,
    best_speed_mps: float,
    expected_azimuth_deg: float | None,
    expected_speed_mps: float | None,
    title: str,
    outfile: Path,
):
    theta = np.deg2rad(azimuth_deg)
    fig, ax = plt.subplots(figsize=(8.2, 7.2), subplot_kw={"projection": "polar"})
    mesh = ax.pcolormesh(theta, speed_mps, score_surface.T, shading="auto", cmap="viridis")
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.plot(np.deg2rad(best_azimuth_deg), best_speed_mps, "*", ms=15, color="white", mec="black", label="Best")
    if expected_azimuth_deg is not None and expected_speed_mps is not None:
        ax.plot(
            np.deg2rad(expected_azimuth_deg), expected_speed_mps,
            "o", ms=7, color="orange", mec="black", label="SLC-40 / provisional speed"
        )
    ax.set_title(title, pad=22)
    ax.legend(loc="upper right", bbox_to_anchor=(1.28, 1.12))
    fig.colorbar(mesh, ax=ax, pad=0.12, label="Joint coherence score")
    fig.savefig(outfile, dpi=220, bbox_inches="tight")
    plt.close(fig)



def boolean_series(values: pd.Series) -> pd.Series:
    """Parse booleans robustly from native bool, numeric, or CSV text columns."""
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False)
    if pd.api.types.is_numeric_dtype(values):
        return values.fillna(0).astype(float).ne(0)
    normalized = values.astype("string").str.strip().str.lower()
    return normalized.isin({"true", "t", "yes", "y", "1"})


def event_score_window(event_number: int) -> tuple[float, float]:
    return tuple(CUSTOM_SCORE_WINDOWS.get(int(event_number), DEFAULT_SCORE_WINDOW_RELATIVE_S))


def wrapped_angle_difference_deg(angle_deg, reference_deg):
    return (np.asarray(angle_deg, dtype=float) - float(reference_deg) + 180.0) % 360.0 - 180.0


def score_surface_constraints(
    azimuth_deg: np.ndarray,
    speed_mps: np.ndarray,
    score_surface: np.ndarray,
    *,
    drops=CONSTRAINT_SCORE_DROPS,
) -> dict[str, float | bool]:
    """Describe surface concentration; widths are descriptive, not confidence intervals."""
    surface = np.asarray(score_surface, dtype=float)
    maximum = float(np.nanmax(surface))
    best_flat = int(np.nanargmax(surface))
    best_az_index, best_speed_index = np.unravel_index(best_flat, surface.shape)
    azimuth_profile = np.nanmax(surface, axis=1)
    speed_profile = np.nanmax(surface, axis=0)
    metrics = {
        "surface_maximum": maximum,
        "surface_median": float(np.nanmedian(surface)),
        "surface_p95": float(np.nanpercentile(surface, 95)),
        "peak_minus_surface_median": maximum - float(np.nanmedian(surface)),
        "peak_minus_surface_p95": maximum - float(np.nanpercentile(surface, 95)),
        "best_on_azimuth_edge": bool(best_az_index in {0, len(azimuth_deg) - 1}),
        "best_on_speed_edge": bool(best_speed_index in {0, len(speed_mps) - 1}),
    }
    azimuth_step = float(np.median(np.diff(azimuth_deg)))
    speed_step = float(np.median(np.diff(speed_mps)))
    for drop in drops:
        suffix = str(drop).replace(".", "p")
        threshold = maximum - float(drop)
        azimuth_keep = np.flatnonzero(azimuth_profile >= threshold)
        speed_keep = np.flatnonzero(speed_profile >= threshold)
        surface_keep = np.isfinite(surface) & (surface >= threshold)
        metrics[f"azimuth_width_drop_{suffix}_deg"] = (
            float(azimuth_deg[azimuth_keep[-1]] - azimuth_deg[azimuth_keep[0]])
            if azimuth_keep.size else np.nan
        )
        metrics[f"speed_width_drop_{suffix}_mps"] = (
            float(speed_mps[speed_keep[-1]] - speed_mps[speed_keep[0]])
            if speed_keep.size else np.nan
        )
        metrics[f"surface_area_drop_{suffix}_deg_mps"] = (
            float(np.count_nonzero(surface_keep) * azimuth_step * speed_step)
        )
    return metrics


def refined_grids(coarse_solution):
    azimuth_grid = np.arange(
        max(BACK_AZIMUTH_MIN_DEG, coarse_solution["best_azimuth_deg"] - FINE_AZIMUTH_HALF_WIDTH_DEG),
        min(BACK_AZIMUTH_MAX_DEG, coarse_solution["best_azimuth_deg"] + FINE_AZIMUTH_HALF_WIDTH_DEG)
        + 0.5 * FINE_AZIMUTH_STEP_DEG,
        FINE_AZIMUTH_STEP_DEG,
    )
    speed_grid = np.arange(
        max(APPARENT_SPEED_MIN_MPS, coarse_solution["best_speed_mps"] - FINE_SPEED_HALF_WIDTH_MPS),
        min(APPARENT_SPEED_MAX_MPS, coarse_solution["best_speed_mps"] + FINE_SPEED_HALF_WIDTH_MPS)
        + 0.5 * FINE_SPEED_STEP_MPS,
        FINE_SPEED_STEP_MPS,
    )
    return azimuth_grid, speed_grid


def refined_polar_plot(
    azimuth_deg,
    speed_mps,
    score_surface,
    *,
    best_azimuth_deg,
    best_speed_mps,
    expected_azimuth_deg,
    expected_speed_mps,
    title,
    outfile,
    zoom=False,
):
    theta = np.deg2rad(azimuth_deg)
    fig, ax = plt.subplots(figsize=(8.2, 7.2), subplot_kw={"projection": "polar"})
    mesh = ax.pcolormesh(theta, speed_mps, np.asarray(score_surface).T, shading="auto", cmap="viridis")
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.plot(np.deg2rad(best_azimuth_deg), best_speed_mps, "*", ms=15, color="white", mec="black", label="Best")
    ax.plot(np.deg2rad(expected_azimuth_deg), expected_speed_mps, "o", ms=7, color="orange", mec="black", label="SLC-40 / 351 m s$^{-1}$")
    if zoom:
        ax.set_thetamin(DISPLAY_AZIMUTH_MIN_DEG)
        ax.set_thetamax(DISPLAY_AZIMUTH_MAX_DEG)
        ax.set_ylim(DISPLAY_SPEED_MIN_MPS, DISPLAY_SPEED_MAX_MPS)
    ax.set_title(title, pad=22)
    ax.legend(loc="upper right", bbox_to_anchor=(1.30, 1.12))
    fig.colorbar(mesh, ax=ax, pad=0.12, label="Joint coherence score")
    fig.savefig(outfile, dpi=220, bbox_inches="tight")
    plt.close(fig)


def rectangular_surface_plot(
    azimuth_deg,
    speed_mps,
    score_surface,
    *,
    best_azimuth_deg,
    best_speed_mps,
    title,
    outfile,
):
    fig, ax = plt.subplots(figsize=(8.8, 5.8), constrained_layout=True)
    mesh = ax.pcolormesh(azimuth_deg, speed_mps, np.asarray(score_surface).T, shading="auto", cmap="viridis")
    ax.plot(best_azimuth_deg, best_speed_mps, "*", ms=14, color="white", mec="black")
    ax.axvline(EXPECTED_SLC40_BACK_AZIMUTH_DEG, color="orange", ls="--", lw=1)
    ax.axhline(PROVISIONAL_EFFECTIVE_SPEED_MPS, color="orange", ls=":", lw=1)
    ax.set(xlabel="Back azimuth (°)", ylabel="Apparent speed (m s$^{-1}$)", title=title)
    fig.colorbar(mesh, ax=ax, label="Joint coherence score")
    fig.savefig(outfile, dpi=220, bbox_inches="tight")
    plt.close(fig)


def event_time_utc(epoch_s):
    return pd.to_datetime(float(epoch_s), unit="s", utc=True)


## Locate Notebook 07 products and normalize their schema

Epoch seconds remain authoritative. The candidate table supplies provisional event
times and links to fixed, unfiltered, moving-median-baseline-corrected event Streams.


In [ ]:
candidate_catalogue_file = (
    CANDIDATE_CATALOGUE_OVERRIDE.resolve()
    if CANDIDATE_CATALOGUE_OVERRIDE is not None
    else first_existing(
        [
            DERIVED_DIR / "candidate_airwave_event_catalogue.csv",
            DERIVED_DIR / "candidate_event_catalogue.csv",
            DERIVED_DIR / "candidate_airwave_events.csv",
        ],
        label="Notebook 07 candidate-event catalogue",
    )
)
event_stream_dir = (
    EVENT_STREAM_DIR_OVERRIDE.resolve()
    if EVENT_STREAM_DIR_OVERRIDE is not None
    else first_existing(
        [EVENT_STREAMS_DIR, candidate_catalogue_file.parent / "event_streams"],
        label="Notebook 07 event-stream directory",
    )
)
candidate_events = normalize_event_catalogue(pd.read_csv(candidate_catalogue_file))
event_time_column = resolve_event_time_column(candidate_events)
print("Candidate catalogue:", candidate_catalogue_file)
print("Event streams:", event_stream_dir)
print("Reference-time column:", event_time_column)
print("Candidate events:", len(candidate_events))
display(candidate_events.head())


## Load surveyed BCHH geometry

Relative plane-wave delays are independent of the arbitrary coordinate origin. The
three-sensor centroid is used here; SLC-40 direction is retained as an external
reference rather than imposed on the inversion.


In [ ]:
geometry_file = (
    GEOMETRY_FILE_OVERRIDE.resolve()
    if GEOMETRY_FILE_OVERRIDE is not None
    else first_existing(
        [
            OTHER_DERIVED_DIR / "bchh_geometry.csv",
        ],
        label="BCHH geometry table",
    )
)

geometry = pd.read_csv(geometry_file)
channel_column = choose_column(geometry, ["channel", "sensor", "label"])
geometry[channel_column] = geometry[channel_column].astype(str).str.upper()

east_column = choose_column(geometry, ["easting", "easting_m", "x_m"], required=False)
north_column = choose_column(geometry, ["northing", "northing_m", "y_m"], required=False)

if east_column is None or north_column is None:
    lon_column = choose_column(geometry, ["longitude", "lon"])
    lat_column = choose_column(geometry, ["latitude", "lat"])
    from pyproj import Transformer
    transformer = Transformer.from_crs("EPSG:4326", "EPSG:32617", always_xy=True)
    east, north = transformer.transform(
        geometry[lon_column].to_numpy(float), geometry[lat_column].to_numpy(float)
    )
    geometry["easting_m"] = east
    geometry["northing_m"] = north
    east_column, north_column = "easting_m", "northing_m"

rows = []
for channel in CHANNELS:
    matches = geometry.loc[geometry[channel_column] == channel]
    if len(matches) != 1:
        raise ValueError(f"Expected one geometry row for {channel}; found {len(matches)}")
    row = matches.iloc[0]
    rows.append({"channel": channel, "east_m": float(row[east_column]), "north_m": float(row[north_column])})

array_geometry = pd.DataFrame(rows).set_index("channel")
array_geometry[["east_relative_m", "north_relative_m"]] = (
    array_geometry[["east_m", "north_m"]]
    - array_geometry[["east_m", "north_m"]].mean(axis=0)
)
print("Geometry:", geometry_file)
display(array_geometry)


## Define permissive analysis and core-catalogue tiers

Every decision is exported. Forced named events remain explicit; they do not silently
pass the short-pulse tests. This permits the main explosion and weak candidates to be
evaluated using array coherence while preserving the original 76-event core label.


In [ ]:
p2p_column = choose_column(candidate_events, [
    "median_stack_peak_to_peak_pa", "stack_peak_to_peak_pa",
    "median_peak_to_peak_pa", "peak_to_peak_pa", "stack_p2p_pa",
])
snr_column = choose_column(candidate_events, [
    "median_stack_snr", "stack_snr", "mean_stack_snr", "snr",
])
channel_count_column = choose_column(
    candidate_events, ["channel_count", "n_channels", "distinct_channel_count"], required=False
)
valid_column = choose_column(
    candidate_events, ["valid_measurement", "measurement_valid", "valid_pair", "measurement_success"], required=False
)

decisions = candidate_events.copy()
decisions["filter_peak_to_peak_pa"] = pd.to_numeric(decisions[p2p_column], errors="coerce")
decisions["filter_stack_snr"] = pd.to_numeric(decisions[snr_column], errors="coerce")
decisions["filter_channel_count"] = (
    pd.to_numeric(decisions[channel_count_column], errors="coerce") if channel_count_column else 3
)
decisions["valid_measurement_boolean"] = (
    boolean_series(decisions[valid_column]) if valid_column else True
)
decisions["forced_named_event"] = decisions["event_number"].isin(FORCE_INCLUDE_EVENT_NUMBERS)
decisions["event_label"] = decisions["event_number"].map(EVENT_LABELS).fillna("short_impulsive_candidate")

decisions["passes_analysis_amplitude"] = decisions["filter_peak_to_peak_pa"] >= ANALYSIS_MIN_STACK_PEAK_TO_PEAK_PA
decisions["passes_analysis_snr"] = decisions["filter_stack_snr"] >= ANALYSIS_MIN_STACK_SNR
decisions["passes_analysis_channel_count"] = decisions["filter_channel_count"] >= ANALYSIS_MIN_CHANNEL_COUNT
decisions["passes_analysis_morphology"] = (
    decisions["valid_measurement_boolean"] if REQUIRE_VALID_MEASUREMENT_FOR_ANALYSIS else True
)
ordinary_analysis_pass = decisions[[
    "passes_analysis_amplitude", "passes_analysis_snr",
    "passes_analysis_channel_count", "passes_analysis_morphology",
]].all(axis=1)
decisions["passes_analysis_filter"] = ordinary_analysis_pass | decisions["forced_named_event"]

decisions["passes_core_amplitude"] = decisions["filter_peak_to_peak_pa"] >= CORE_MIN_STACK_PEAK_TO_PEAK_PA
decisions["passes_core_snr"] = decisions["filter_stack_snr"] >= CORE_MIN_STACK_SNR
decisions["passes_core_channel_count"] = decisions["filter_channel_count"] >= CORE_MIN_CHANNEL_COUNT
decisions["passes_core_filter"] = decisions[[
    "passes_core_amplitude", "passes_core_snr", "passes_core_channel_count",
]].all(axis=1) & decisions["valid_measurement_boolean"]

def analysis_reason(row):
    if row["forced_named_event"]:
        return "forced_named_complex_event"
    failed = []
    if not row["passes_analysis_amplitude"]: failed.append("peak_to_peak")
    if not row["passes_analysis_snr"]: failed.append("stack_snr")
    if not row["passes_analysis_channel_count"]: failed.append("channel_count")
    if not row["passes_analysis_morphology"]: failed.append("short_pulse_morphology")
    return "pass" if not failed else ",".join(failed)

decisions["analysis_filter_reason"] = decisions.apply(analysis_reason, axis=1)
decisions.to_csv(OUTPUT_ROOT / "event_filter_decisions.csv", index=False)

selected_events = decisions.loc[decisions["passes_analysis_filter"]].copy()
if MAX_EVENTS is not None:
    selected_events = selected_events.head(MAX_EVENTS).copy()

print(f"Permissive analysis set: {len(selected_events)} / {len(decisions)}")
print(f"Original core label: {int(decisions['passes_core_filter'].sum())} / {len(decisions)}")
display(decisions[[
    "event_number", "event_label", "filter_peak_to_peak_pa", "filter_stack_snr",
    "passes_analysis_filter", "passes_core_filter", "analysis_filter_reason",
]])


## Coarse-to-fine joint plane-wave search

For each trial, all three traces are shifted using a physically valid slowness vector,
trimmed to the same overlap, normalized, and scored jointly. The broad surface is
retained for composite maps. A fine event-specific surface supplies the reported
maximum and descriptive peak-width metrics.


In [ ]:
def planar_relative_delays(back_azimuth_deg: float, speed_mps: float) -> dict[str, float]:
    theta = np.deg2rad(back_azimuth_deg)
    propagation_east = -np.sin(theta)
    propagation_north = -np.cos(theta)
    projected = (
        array_geometry["east_relative_m"].to_numpy() * propagation_east
        + array_geometry["north_relative_m"].to_numpy() * propagation_north
    )
    delays = projected / speed_mps
    delays = delays - delays[list(CHANNELS).index("HD2")]
    return dict(zip(CHANNELS, delays))


def search_planar_surface(stream, reference_epoch_s, window_relative_s, azimuth_grid, speed_grid):
    score = np.full((len(azimuth_grid), len(speed_grid)), np.nan)
    semblance = np.full_like(score, np.nan)
    mean_corr = np.full_like(score, np.nan)
    for az_index, back_azimuth in enumerate(azimuth_grid):
        for speed_index, speed in enumerate(speed_grid):
            delays = planar_relative_delays(float(back_azimuth), float(speed))
            _, aligned = aligned_waveforms(
                stream,
                reference_epoch_s=reference_epoch_s,
                relative_delays_s=delays,
                window_relative_s=window_relative_s,
            )
            metrics = coherence_metrics(aligned, score_weight_semblance=SCORE_WEIGHT_SEMBLANCE)
            score[az_index, speed_index] = metrics["coherence_score"]
            semblance[az_index, speed_index] = metrics["semblance"]
            mean_corr[az_index, speed_index] = metrics["mean_pairwise_correlation"]
    if not np.isfinite(score).any():
        raise ValueError("Complete planar surface is non-finite; check event time/window coverage")
    best_flat = int(np.nanargmax(score))
    best_az_index, best_speed_index = np.unravel_index(best_flat, score.shape)
    return {
        "score": score,
        "semblance": semblance,
        "mean_corr": mean_corr,
        "best_azimuth_deg": float(azimuth_grid[best_az_index]),
        "best_speed_mps": float(speed_grid[best_speed_index]),
        "best_score": float(score[best_az_index, best_speed_index]),
        "best_semblance": float(semblance[best_az_index, best_speed_index]),
        "best_mean_corr": float(mean_corr[best_az_index, best_speed_index]),
    }


def stability_solutions(stream, reference_epoch_s, main_solution, base_window):
    if not RUN_WINDOW_STABILITY:
        return pd.DataFrame()
    rows = []
    base_start, base_end = base_window
    center = 0.5 * (base_start + base_end)
    duration = base_end - base_start
    azimuth_grid = np.arange(
        max(BACK_AZIMUTH_MIN_DEG, main_solution["best_azimuth_deg"] - STABILITY_LOCAL_AZIMUTH_HALF_WIDTH_DEG),
        min(BACK_AZIMUTH_MAX_DEG, main_solution["best_azimuth_deg"] + STABILITY_LOCAL_AZIMUTH_HALF_WIDTH_DEG)
        + 0.5 * STABILITY_AZIMUTH_STEP_DEG,
        STABILITY_AZIMUTH_STEP_DEG,
    )
    speed_grid = np.arange(
        max(APPARENT_SPEED_MIN_MPS, main_solution["best_speed_mps"] - STABILITY_LOCAL_SPEED_HALF_WIDTH_MPS),
        min(APPARENT_SPEED_MAX_MPS, main_solution["best_speed_mps"] + STABILITY_LOCAL_SPEED_HALF_WIDTH_MPS)
        + 0.5 * STABILITY_SPEED_STEP_MPS,
        STABILITY_SPEED_STEP_MPS,
    )
    for shift_s in STABILITY_WINDOW_SHIFTS_S:
        for factor in STABILITY_DURATION_FACTORS:
            half = 0.5 * duration * factor
            window = (center + shift_s - half, center + shift_s + half)
            solution = search_planar_surface(stream, reference_epoch_s, window, azimuth_grid, speed_grid)
            rows.append({
                "window_shift_s": shift_s,
                "duration_factor": factor,
                "window_start_s": window[0],
                "window_end_s": window[1],
                "back_azimuth_deg": solution["best_azimuth_deg"],
                "apparent_speed_mps": solution["best_speed_mps"],
                "coherence_score": solution["best_score"],
            })
    return pd.DataFrame(rows)


def provisional_quality_class(row):
    if (
        row["maximum_coherence_score"] >= QUALITY_A_MIN_SCORE
        and row["stability_back_azimuth_std_deg"] <= QUALITY_A_MAX_AZIMUTH_STD_DEG
        and row["stability_speed_std_mps"] <= QUALITY_A_MAX_SPEED_STD_MPS
    ):
        return "A_high_quality"
    if (
        row["maximum_coherence_score"] >= QUALITY_B_MIN_SCORE
        and row["stability_back_azimuth_std_deg"] <= QUALITY_B_MAX_AZIMUTH_STD_DEG
        and row["stability_speed_std_mps"] <= QUALITY_B_MAX_SPEED_STD_MPS
    ):
        return "B_probable"
    return "C_weak_or_broad"


## Run the refined event analysis

Each event writes a broad common-grid surface and a fine local surface. Pairwise
correlations include an HD1 deficit diagnostic because HD2–HD3 were previously found
to agree substantially better than either HD1 pair.


In [ ]:
coarse_azimuth_grid = np.arange(
    BACK_AZIMUTH_MIN_DEG, BACK_AZIMUTH_MAX_DEG + 0.5 * BACK_AZIMUTH_STEP_DEG, BACK_AZIMUTH_STEP_DEG
)
coarse_speed_grid = np.arange(
    APPARENT_SPEED_MIN_MPS, APPARENT_SPEED_MAX_MPS + 0.5 * APPARENT_SPEED_STEP_MPS, APPARENT_SPEED_STEP_MPS
)

result_rows = []
stability_frames = []
pairwise_rows = []
coarse_surface_records = []

for progress, (_, event_row) in enumerate(selected_events.iterrows(), start=1):
    event_number = int(event_row["event_number"])
    label = str(event_row["event_label"])
    reference_epoch_s = float(event_row[event_time_column])
    score_window = event_score_window(event_number)
    stream_path, stream = load_event_stream(event_row, event_stream_dir)

    coarse = search_planar_surface(
        stream, reference_epoch_s, score_window, coarse_azimuth_grid, coarse_speed_grid
    )
    fine_azimuth_grid, fine_speed_grid = refined_grids(coarse)
    fine = search_planar_surface(
        stream, reference_epoch_s, score_window, fine_azimuth_grid, fine_speed_grid
    )
    constraints = score_surface_constraints(fine_azimuth_grid, fine_speed_grid, fine["score"])

    best_delays = planar_relative_delays(fine["best_azimuth_deg"], fine["best_speed_mps"])
    relative_time, best_aligned = aligned_waveforms(
        stream,
        reference_epoch_s=reference_epoch_s,
        relative_delays_s=best_delays,
        window_relative_s=score_window,
    )
    corr_matrix, residual_lag_matrix, closure_s = pairwise_diagnostics(
        best_aligned, common_sampling_rate(stream), MAX_RESIDUAL_LAG_S
    )
    corr12, corr13, corr23 = corr_matrix[0, 1], corr_matrix[0, 2], corr_matrix[1, 2]
    mean_hd1_corr = float(np.nanmean([corr12, corr13]))
    hd1_deficit = float(corr23 - mean_hd1_corr)

    stability = stability_solutions(stream, reference_epoch_s, fine, score_window)
    if not stability.empty:
        stability.insert(0, "event_number", event_number)
        stability_frames.append(stability)
        azimuth_std = float(stability["back_azimuth_deg"].std(ddof=0))
        speed_std = float(stability["apparent_speed_mps"].std(ddof=0))
        stability_score_min = float(stability["coherence_score"].min())
    else:
        azimuth_std = speed_std = stability_score_min = np.nan

    row = {
        "event_number": event_number,
        "event_label": label,
        "reference_time_epoch_s": reference_epoch_s,
        "reference_time_utc": event_time_utc(reference_epoch_s),
        "stream_file": str(stream_path),
        "score_window_start_s": score_window[0],
        "score_window_end_s": score_window[1],
        "passes_core_filter": bool(event_row["passes_core_filter"]),
        "forced_named_event": bool(event_row["forced_named_event"]),
        "coarse_back_azimuth_deg": coarse["best_azimuth_deg"],
        "coarse_apparent_speed_mps": coarse["best_speed_mps"],
        "best_back_azimuth_deg": fine["best_azimuth_deg"],
        "best_apparent_speed_mps": fine["best_speed_mps"],
        "maximum_coherence_score": fine["best_score"],
        "best_semblance": fine["best_semblance"],
        "best_mean_pairwise_correlation": fine["best_mean_corr"],
        "correlation_hd1_hd2": corr12,
        "correlation_hd1_hd3": corr13,
        "correlation_hd2_hd3": corr23,
        "mean_hd1_pair_correlation": mean_hd1_corr,
        "hd1_correlation_deficit": hd1_deficit,
        "residual_lag_closure_s": closure_s,
        "stability_back_azimuth_std_deg": azimuth_std,
        "stability_speed_std_mps": speed_std,
        "stability_score_min": stability_score_min,
        "stack_peak_to_peak_pa": float(np.nanmax(np.nanmedian(best_aligned, axis=0)) - np.nanmin(np.nanmedian(best_aligned, axis=0))),
        "hd1_delay_relative_hd2_s": best_delays["HD1"],
        "hd3_delay_relative_hd2_s": best_delays["HD3"],
        **constraints,
    }
    result_rows.append(row)
    coarse_surface_records.append({
        "event_number": event_number,
        "passes_core_filter": bool(event_row["passes_core_filter"]),
        "score": coarse["score"],
        "maximum": coarse["best_score"],
    })

    for i, channel_i in enumerate(CHANNELS):
        for j, channel_j in enumerate(CHANNELS):
            pairwise_rows.append({
                "event_number": event_number,
                "channel_i": channel_i,
                "channel_j": channel_j,
                "best_aligned_correlation": corr_matrix[i, j],
                "residual_lag_s": residual_lag_matrix[i, j],
            })

    np.savez_compressed(
        SURFACE_DIR / f"event_{event_number:03d}_planar_surface.npz",
        coarse_back_azimuth_deg=coarse_azimuth_grid,
        coarse_apparent_speed_mps=coarse_speed_grid,
        coarse_coherence_score=coarse["score"],
        coarse_semblance=coarse["semblance"],
        coarse_mean_pairwise_correlation=coarse["mean_corr"],
        fine_back_azimuth_deg=fine_azimuth_grid,
        fine_apparent_speed_mps=fine_speed_grid,
        fine_coherence_score=fine["score"],
        fine_semblance=fine["semblance"],
        fine_mean_pairwise_correlation=fine["mean_corr"],
    )

    refined_polar_plot(
        coarse_azimuth_grid, coarse_speed_grid, coarse["score"],
        best_azimuth_deg=fine["best_azimuth_deg"], best_speed_mps=fine["best_speed_mps"],
        expected_azimuth_deg=EXPECTED_SLC40_BACK_AZIMUTH_DEG,
        expected_speed_mps=PROVISIONAL_EFFECTIVE_SPEED_MPS,
        title=f"Event {event_number:03d}: planar coherence (broad search)",
        outfile=FIGURE_DIR / f"event_{event_number:03d}_polar_broad.png",
    )
    refined_polar_plot(
        fine_azimuth_grid, fine_speed_grid, fine["score"],
        best_azimuth_deg=fine["best_azimuth_deg"], best_speed_mps=fine["best_speed_mps"],
        expected_azimuth_deg=EXPECTED_SLC40_BACK_AZIMUTH_DEG,
        expected_speed_mps=PROVISIONAL_EFFECTIVE_SPEED_MPS,
        title=f"Event {event_number:03d}: refined planar coherence",
        outfile=FIGURE_DIR / f"event_{event_number:03d}_polar_refined.png",
        zoom=True,
    )
    rectangular_surface_plot(
        fine_azimuth_grid, fine_speed_grid, fine["score"],
        best_azimuth_deg=fine["best_azimuth_deg"], best_speed_mps=fine["best_speed_mps"],
        title=f"Event {event_number:03d}: refined azimuth–speed surface",
        outfile=FIGURE_DIR / f"event_{event_number:03d}_surface_refined.png",
    )

    fig, axes = plt.subplots(4, 1, figsize=(10.5, 8.5), sharex=True, constrained_layout=True)
    for index, channel in enumerate(CHANNELS):
        axes[index].plot(relative_time, best_aligned[index], lw=1.0, color=f"C{index}")
        axes[index].axhline(0, color="0.6", lw=0.5)
        axes[index].set_ylabel(f"{channel}\nPa")
    axes[3].plot(relative_time, np.nanmedian(best_aligned, axis=0), color="black", lw=1.2)
    axes[3].axhline(0, color="0.6", lw=0.5)
    axes[3].set_ylabel("Median\nstack (Pa)")
    axes[3].set_xlabel("Time relative to reference event time (s)")
    fig.suptitle(
        f"Event {event_number:03d} ({label}): refined planar alignment\n"
        f"BAZ={fine['best_azimuth_deg']:.2f}°, speed={fine['best_speed_mps']:.1f} m s⁻¹, score={fine['best_score']:.3f}"
    )
    fig.savefig(FIGURE_DIR / f"event_{event_number:03d}_aligned.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    print(
        f"[{progress:03d}/{len(selected_events):03d}] event {event_number:03d}: "
        f"BAZ={fine['best_azimuth_deg']:.2f}°, v={fine['best_speed_mps']:.1f} m/s, "
        f"score={fine['best_score']:.3f}, HD1 deficit={hd1_deficit:.3f}"
    )

planar_results = pd.DataFrame(result_rows).sort_values("event_number").reset_index(drop=True)
planar_results["solution_quality_class"] = planar_results.apply(provisional_quality_class, axis=1)
planar_stability = pd.concat(stability_frames, ignore_index=True) if stability_frames else pd.DataFrame()
planar_pairwise = pd.DataFrame(pairwise_rows)

planar_results.to_csv(OUTPUT_ROOT / "planar_array_results.csv", index=False)
planar_stability.to_csv(OUTPUT_ROOT / "planar_window_stability.csv", index=False)
planar_pairwise.to_csv(OUTPUT_ROOT / "planar_pairwise_diagnostics.csv", index=False)
display(planar_results)


## Principal/first-stage explosion

The principal explosion is evaluated using the same plane-wave model but a broader
scoring interval that captures its coherent compression and long positive decay. Its
inclusion is independent of the short N-wave-like detector. The resulting direction
is used below as an empirical reference for relative source movement.


In [ ]:
principal_result = planar_results.loc[
    planar_results["event_number"] == PRINCIPAL_EXPLOSION_EVENT_NUMBER
].copy()
if principal_result.empty:
    raise ValueError("Principal explosion was not analyzed; check force-include configuration")
principal_result.to_csv(OUTPUT_ROOT / "principal_explosion_planar_result.csv", index=False)
display(principal_result.T)
print(
    "Principal explosion reference: "
    f"BAZ={principal_result.iloc[0]['best_back_azimuth_deg']:.2f}°, "
    f"speed={principal_result.iloc[0]['best_apparent_speed_mps']:.1f} m/s, "
    f"coherence={principal_result.iloc[0]['maximum_coherence_score']:.3f}"
)


## Composite planar coherence maps

All broad surfaces share one grid. We save arithmetic mean, median, quality-weighted
mean, and event-normalized mean composites. Coherence is not a probability, so
surfaces are not multiplied or described as formal likelihoods.


In [ ]:
def build_composites(records, label):
    if not records:
        return None
    stack = np.stack([record["score"] for record in records])
    maxima = np.asarray([record["maximum"] for record in records], dtype=float)
    normalized = []
    for surface in stack:
        baseline = np.nanmedian(surface)
        scale = np.nanmax(surface) - baseline
        normalized.append((surface - baseline) / scale if scale > 0 else np.zeros_like(surface))
    normalized = np.stack(normalized)
    composites = {
        "mean": np.nanmean(stack, axis=0),
        "median": np.nanmedian(stack, axis=0),
        "weighted_mean": np.average(stack, axis=0, weights=maxima),
        "normalized_mean": np.nanmean(normalized, axis=0),
    }
    np.savez_compressed(
        COMPOSITE_DIR / f"composite_{label}.npz",
        back_azimuth_deg=coarse_azimuth_grid,
        apparent_speed_mps=coarse_speed_grid,
        event_numbers=np.asarray([record["event_number"] for record in records]),
        **composites,
    )
    for method, surface in composites.items():
        best_flat = int(np.nanargmax(surface))
        az_index, speed_index = np.unravel_index(best_flat, surface.shape)
        refined_polar_plot(
            coarse_azimuth_grid, coarse_speed_grid, surface,
            best_azimuth_deg=float(coarse_azimuth_grid[az_index]),
            best_speed_mps=float(coarse_speed_grid[speed_index]),
            expected_azimuth_deg=EXPECTED_SLC40_BACK_AZIMUTH_DEG,
            expected_speed_mps=PROVISIONAL_EFFECTIVE_SPEED_MPS,
            title=f"{label.replace('_', ' ').title()}: {method.replace('_', ' ')} composite",
            outfile=COMPOSITE_DIR / f"composite_{label}_{method}_polar.png",
            zoom=True,
        )
        rectangular_surface_plot(
            coarse_azimuth_grid, coarse_speed_grid, surface,
            best_azimuth_deg=float(coarse_azimuth_grid[az_index]),
            best_speed_mps=float(coarse_speed_grid[speed_index]),
            title=f"{label.replace('_', ' ').title()}: {method.replace('_', ' ')} composite",
            outfile=COMPOSITE_DIR / f"composite_{label}_{method}_rectangular.png",
        )
    return composites

all_composites = build_composites(coarse_surface_records, "all_analyzed")
core_composites = build_composites(
    [record for record in coarse_surface_records if record["passes_core_filter"]], "core_catalogue"
)
high_quality_numbers = set(
    planar_results.loc[planar_results["solution_quality_class"] == "A_high_quality", "event_number"]
)
high_quality_composites = build_composites(
    [record for record in coarse_surface_records if record["event_number"] in high_quality_numbers],
    "high_quality",
)
print("Composite maps written to", COMPOSITE_DIR)


## Relative back azimuth and cross-range displacement through time

Relative direction is referenced to the measured principal explosion rather than the
nominal SLC-40 azimuth. This removes a common systematic offset before comparing
events. The right axis is an exact transformation at 1.42 km, not an independent
second dataset.


In [ ]:
principal_back_azimuth_deg = float(principal_result.iloc[0]["best_back_azimuth_deg"])
sequence_start_epoch_s = float(planar_results["reference_time_epoch_s"].min())
planar_results["sequence_time_s"] = planar_results["reference_time_epoch_s"] - sequence_start_epoch_s
planar_results["relative_back_azimuth_deg"] = wrapped_angle_difference_deg(
    planar_results["best_back_azimuth_deg"], principal_back_azimuth_deg
)
planar_results["approximate_cross_range_m"] = ARRAY_TO_SOURCE_REFERENCE_RANGE_M * np.tan(
    np.deg2rad(planar_results["relative_back_azimuth_deg"])
)
planar_results.to_csv(OUTPUT_ROOT / "planar_array_results.csv", index=False)

def degrees_to_metres(angle_deg):
    return ARRAY_TO_SOURCE_REFERENCE_RANGE_M * np.tan(np.deg2rad(angle_deg))

def metres_to_degrees(distance_m):
    return np.rad2deg(np.arctan(np.asarray(distance_m) / ARRAY_TO_SOURCE_REFERENCE_RANGE_M))

fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
scatter = ax.scatter(
    planar_results["sequence_time_s"],
    planar_results["relative_back_azimuth_deg"],
    c=planar_results["maximum_coherence_score"],
    cmap="viridis", vmin=0, vmax=1, s=35,
)
if RUN_WINDOW_STABILITY:
    ax.errorbar(
        planar_results["sequence_time_s"],
        planar_results["relative_back_azimuth_deg"],
        yerr=planar_results["stability_back_azimuth_std_deg"],
        fmt="none", ecolor="0.5", alpha=0.45, lw=0.8,
    )
ax.axhline(0, color="orange", ls="--", lw=1, label="Principal-explosion direction")
ax.set(
    xlabel="Time since first analyzed candidate (s)",
    ylabel="Back-azimuth change relative to principal explosion (°)",
    title="Relative source direction through the explosion sequence",
)
secondary = ax.secondary_yaxis("right", functions=(degrees_to_metres, metres_to_degrees))
secondary.set_ylabel("Approximate transverse displacement at 1.42 km (m)")
ax.legend()
fig.colorbar(scatter, ax=ax, label="Maximum coherence score")
fig.savefig(OUTPUT_ROOT / "relative_back_azimuth_cross_range_vs_time.png", dpi=240, bbox_inches="tight")
plt.show()

display(planar_results[[
    "event_number", "reference_time_utc", "best_back_azimuth_deg",
    "relative_back_azimuth_deg", "approximate_cross_range_m",
    "best_apparent_speed_mps", "maximum_coherence_score", "solution_quality_class",
]])


## Catalogue summaries and Notebook 09 handoff

Inspect continuous quality metrics rather than treating the provisional A/B/C labels
as immutable scientific classes. The refined planar results remain the direct input
to fixed-source and local-grid spherical analyses.


In [ ]:
if not planar_results.empty:
    fig, axes = plt.subplots(4, 1, figsize=(12, 11), sharex=True, constrained_layout=True)
    axes[0].scatter(planar_results["event_number"], planar_results["best_back_azimuth_deg"], c=planar_results["maximum_coherence_score"], cmap="viridis")
    axes[0].axhline(EXPECTED_SLC40_BACK_AZIMUTH_DEG, color="orange", ls="--")
    axes[0].set_ylabel("Back azimuth (°)")
    axes[1].scatter(planar_results["event_number"], planar_results["best_apparent_speed_mps"], c=planar_results["maximum_coherence_score"], cmap="viridis")
    axes[1].axhline(PROVISIONAL_EFFECTIVE_SPEED_MPS, color="orange", ls="--")
    axes[1].set_ylabel("Speed (m s⁻¹)")
    axes[2].scatter(planar_results["event_number"], planar_results["maximum_coherence_score"], color="black", s=18)
    axes[2].set_ylabel("Coherence")
    axes[3].scatter(planar_results["event_number"], planar_results["hd1_correlation_deficit"], c=planar_results["maximum_coherence_score"], cmap="viridis", s=22)
    axes[3].axhline(0, color="0.5", ls="--")
    axes[3].set_ylabel("HD1 correlation deficit")
    axes[3].set_xlabel("Candidate event number")
    fig.savefig(OUTPUT_ROOT / "planar_catalogue_summary.png", dpi=220, bbox_inches="tight")
    plt.show()

print("Solution-quality classes:")
display(planar_results["solution_quality_class"].value_counts().rename_axis("class").to_frame("events"))
print("Median refined speed:", planar_results["best_apparent_speed_mps"].median(), "m/s")
print("Median refined back azimuth:", planar_results["best_back_azimuth_deg"].median(), "degrees")

processing_metadata = {
    "candidate_catalogue_file": str(candidate_catalogue_file),
    "event_stream_dir": str(event_stream_dir),
    "geometry_file": str(geometry_file),
    "analysis_thresholds": {
        "minimum_stack_peak_to_peak_pa": ANALYSIS_MIN_STACK_PEAK_TO_PEAK_PA,
        "minimum_stack_snr": ANALYSIS_MIN_STACK_SNR,
        "minimum_channel_count": ANALYSIS_MIN_CHANNEL_COUNT,
    },
    "core_thresholds": {
        "minimum_stack_peak_to_peak_pa": CORE_MIN_STACK_PEAK_TO_PEAK_PA,
        "minimum_stack_snr": CORE_MIN_STACK_SNR,
        "minimum_channel_count": CORE_MIN_CHANNEL_COUNT,
    },
    "force_include_event_numbers": list(FORCE_INCLUDE_EVENT_NUMBERS),
    "custom_score_windows": {str(key): list(value) for key, value in CUSTOM_SCORE_WINDOWS.items()},
    "coarse_back_azimuth_grid_deg": [BACK_AZIMUTH_MIN_DEG, BACK_AZIMUTH_MAX_DEG, BACK_AZIMUTH_STEP_DEG],
    "coarse_apparent_speed_grid_mps": [APPARENT_SPEED_MIN_MPS, APPARENT_SPEED_MAX_MPS, APPARENT_SPEED_STEP_MPS],
    "fine_grid": {
        "azimuth_half_width_deg": FINE_AZIMUTH_HALF_WIDTH_DEG,
        "azimuth_step_deg": FINE_AZIMUTH_STEP_DEG,
        "speed_half_width_mps": FINE_SPEED_HALF_WIDTH_MPS,
        "speed_step_mps": FINE_SPEED_STEP_MPS,
    },
    "score_weight_semblance": SCORE_WEIGHT_SEMBLANCE,
    "back_azimuth_convention": "array toward source, clockwise from north",
    "constraint_width_warning": "Near-maximum widths are descriptive, not formal confidence intervals.",
    "relative_azimuth_reference_event": PRINCIPAL_EXPLOSION_EVENT_NUMBER,
    "cross_range_reference_distance_m": ARRAY_TO_SOURCE_REFERENCE_RANGE_M,
}
(OUTPUT_ROOT / "planar_processing_metadata.json").write_text(json.dumps(processing_metadata, indent=2))
print("Notebook 09 input:", OUTPUT_ROOT / "planar_array_results.csv")
